# Day 8 · Lab 5: Docker Containerization & Multi-Container Runtime
### Module 7: AI Pair Programming and Agentic Coding · Udaan Learning

---

## 🎯 Learning Objectives
1. Understand why Docker is mandatory for AI engineering (**eliminating "works on my machine"**).
2. Write a **Production Multi-Stage Dockerfile** that optimizes image size from 1.5GB to 120MB.
3. Orchestrate multiple services using **`docker-compose.yml`** (FastAPI app + Redis cache + PostgreSQL/pgvector).
4. Master essential Docker CLI commands.



In [ ]:
import os
import json

print("[*] Lab 5: Docker & Containerization initialized.")


---
## 1. Production Multi-Stage Dockerfile
Multi-stage builds separate the build environment (compilers, headers) from the final runtime environment:



In [ ]:
dockerfile_content = '''# Stage 1: Build & Dependencies Stage
FROM python:3.12-slim AS builder
WORKDIR /app

RUN apt-get update && apt-get install -y --no-install-recommends gcc build-essential && rm -rf /var/lib/apt/lists/*

COPY requirements.txt .
RUN pip install --no-cache-dir --user -r requirements.txt

# Stage 2: Minimal Production Runtime Stage
FROM python:3.12-slim AS runner
WORKDIR /app

# Copy only installed wheels from builder stage
COPY --from=builder /root/.local /root/.local
COPY . .

ENV PATH=/root/.local/bin:$PATH
ENV PYTHONUNBUFFERED=1

EXPOSE 8000
HEALTHCHECK --interval=30s --timeout=5s --start-period=5s --retries=3 \
  CMD curl -f http://localhost:8000/health || exit 1

CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]
'''

print("=== PRODUCTION MULTI-STAGE DOCKERFILE ===")
print(dockerfile_content)


---
## 2. Multi-Container Composition (`docker-compose.yml`)
AI microservices rarely run alone. They require a cache (Redis) and a database (PostgreSQL/pgvector):



In [ ]:
docker_compose_content = '''version: "3.8"

services:
  ai-api:
    build: .
    ports:
      - "8000:8000"
    environment:
      - DATABASE_URL=postgresql://postgres:postgres@db:5432/healthcare
      - REDIS_URL=redis://cache:6379/0
      - GEMINI_API_KEY=${GEMINI_API_KEY}
    depends_on:
      - db
      - cache
    restart: unless-stopped

  db:
    image: pgvector/pgvector:pg16
    environment:
      - POSTGRES_DB=healthcare
      - POSTGRES_USER=postgres
      - POSTGRES_PASSWORD=postgres
    volumes:
      - pgdata:/var/lib/postgresql/data
    ports:
      - "5432:5432"

  cache:
    image: redis:7-alpine
    ports:
      - "6379:6379"

volumes:
  pgdata:
'''

print("=== DOCKER COMPOSE CONFIGURATION ===")
print(docker_compose_content)


---
## 3. Essential Docker CLI Commands
| Action | Command | Purpose |
| :--- | :--- | :--- |
| **Build image** | `docker build -t ai-service:latest .` | Compiles Dockerfile into runnable image |
| **Run container** | `docker run -d -p 8000:8000 --env-file .env ai-service` | Launches container in background |
| **Inspect logs** | `docker logs -f <container_id>` | Streams live stdout/stderr logs |
| **Compose start** | `docker compose up -d` | Spins up multi-container cluster |
| **Compose stop** | `docker compose down -v` | Stops containers and removes temporary volumes |



---
## 💡 Summary
- Multi-stage Docker builds keep production containers lightweight and secure.
- Docker Compose orchestrates dependencies (database, Redis, microservice) in 1 command.
- In **Lab 6**, we build **GitHub Actions CI/CD & Production LLMOps Pipelines**!

